In [2]:
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# Load gold risk table — this has all features + label
df = spark.table("gold_student_risk")

# Features we use:
# - num_of_prev_attempts  (numeric)
# - studied_credits       (numeric)
# - avg_score             (numeric)
# - total_clicks          (numeric)
# - gender                (categorical)
# - age_band              (categorical)
# - highest_education     (categorical)

# Label: is_at_risk (1 = Withdrawn or Fail, 0 = Pass or Distinction)

# Drop rows with nulls in any feature column
model_df = df.select(
    "student_id",
    "num_of_prev_attempts",
    "studied_credits",
    "avg_score",
    "total_clicks",
    "gender",
    "age_band",
    "highest_education",
    "is_at_risk"
).dropna()

print(f"Model dataset: {model_df.count():,} rows")
model_df.groupBy("is_at_risk").count().show()

StatementMeta(, eb7d613a-2ffc-4468-b31c-e227f4514645, 4, Finished, Available, Finished, False)

Model dataset: 28,785 rows
+----------+-----+
|is_at_risk|count|
+----------+-----+
|         1|15307|
|         0|13478|
+----------+-----+



In [3]:
# String indexers convert categorical text to numbers
# e.g. "M"/"F" → 0/1
gender_idx     = StringIndexer(inputCol="gender",            outputCol="gender_idx",     handleInvalid="keep")
age_idx        = StringIndexer(inputCol="age_band",          outputCol="age_idx",         handleInvalid="keep")
education_idx  = StringIndexer(inputCol="highest_education", outputCol="education_idx",   handleInvalid="keep")

# Assemble all features into one vector column
assembler = VectorAssembler(
    inputCols=[
        "num_of_prev_attempts",
        "studied_credits",
        "avg_score",
        "total_clicks",
        "gender_idx",
        "age_idx",
        "education_idx"
    ],
    outputCol="features_raw"
)

# Scale features so no single column dominates
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withStd=True, withMean=True)

# Logistic Regression — interpretable, fast, good baseline
lr = LogisticRegression(
    featuresCol="features",
    labelCol="is_at_risk",
    maxIter=100,
    regParam=0.01
)

pipeline = Pipeline(stages=[gender_idx, age_idx, education_idx, assembler, scaler, lr])

print("Pipeline defined")

StatementMeta(, eb7d613a-2ffc-4468-b31c-e227f4514645, 5, Finished, Available, Finished, False)

Pipeline defined


In [4]:
# 80% train, 20% test — stratified by label using random seed for reproducibility
train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=42)

print(f"Train: {train_df.count():,} | Test: {test_df.count():,}")

# Fit the pipeline on training data
model = pipeline.fit(train_df)

print("Model trained")

StatementMeta(, eb7d613a-2ffc-4468-b31c-e227f4514645, 6, Finished, Available, Finished, False)

Train: 23,059 | Test: 5,726


Model trained


In [5]:
# Generate predictions on test set
predictions = model.transform(test_df)

# AUC-ROC — measures how well model separates at-risk vs not-at-risk
# 0.5 = random, 1.0 = perfect
auc_evaluator = BinaryClassificationEvaluator(
    labelCol="is_at_risk",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
auc = auc_evaluator.evaluate(predictions)

# Accuracy
acc_evaluator = MulticlassClassificationEvaluator(
    labelCol="is_at_risk",
    predictionCol="prediction",
    metricName="accuracy"
)
accuracy = acc_evaluator.evaluate(predictions)

# F1 score — balances precision and recall, better than accuracy for imbalanced labels
f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="is_at_risk",
    predictionCol="prediction",
    metricName="f1"
)
f1 = f1_evaluator.evaluate(predictions)

print(f"--- Model Evaluation ---")
print(f"AUC-ROC:  {auc:.4f}   (target: > 0.70)")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")

# Confusion matrix
print("\n--- Confusion Matrix ---")
predictions.groupBy("is_at_risk", "prediction").count().orderBy("is_at_risk", "prediction").show()

StatementMeta(, eb7d613a-2ffc-4468-b31c-e227f4514645, 7, Finished, Available, Finished, False)

--- Model Evaluation ---
AUC-ROC:  0.8729   (target: > 0.70)
Accuracy: 0.7847
F1 Score: 0.7848

--- Confusion Matrix ---
+----------+----------+-----+
|is_at_risk|prediction|count|
+----------+----------+-----+
|         0|       0.0| 2194|
|         0|       1.0|  494|
|         1|       0.0|  739|
|         1|       1.0| 2299|
+----------+----------+-----+



In [6]:
# Extract probability of being at risk (class 1 probability)
from pyspark.ml.functions import vector_to_array

predictions_out = predictions \
    .withColumn("prob_array", vector_to_array(F.col("probability"))) \
    .withColumn("risk_probability", F.round(F.col("prob_array")[1], 4)) \
    .withColumn("predicted_at_risk", F.col("prediction").cast("integer")) \
    .withColumn("_model_version", F.lit("logistic_regression_v1")) \
    .withColumn("_scored_at", F.current_timestamp()) \
    .select(
        "student_id",
        "avg_score",
        "total_clicks",
        "num_of_prev_attempts",
        "studied_credits",
        "is_at_risk",
        "predicted_at_risk",
        "risk_probability",
        "_model_version",
        "_scored_at"
    )

predictions_out.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_student_risk_predictions")

print(f"✅ gold_student_risk_predictions: {predictions_out.count():,} rows")

# Distribution of predicted risk probabilities
predictions_out.select(
    F.round(F.avg("risk_probability"), 4).alias("avg_risk_prob"),
    F.round(F.min("risk_probability"), 4).alias("min_risk_prob"),
    F.round(F.max("risk_probability"), 4).alias("max_risk_prob"),
).show()

StatementMeta(, eb7d613a-2ffc-4468-b31c-e227f4514645, 8, Finished, Available, Finished, False)

✅ gold_student_risk_predictions: 5,726 rows
+-------------+-------------+-------------+
|avg_risk_prob|min_risk_prob|max_risk_prob|
+-------------+-------------+-------------+
|       0.5337|          0.0|       0.9934|
+-------------+-------------+-------------+



In [7]:
# Extract the logistic regression stage from the pipeline
lr_model = model.stages[-1]

feature_names = [
    "num_of_prev_attempts",
    "studied_credits",
    "avg_score",
    "total_clicks",
    "gender",
    "age_band",
    "highest_education"
]

print("--- Feature Coefficients (higher absolute value = more influential) ---")
for name, coef in sorted(zip(feature_names, lr_model.coefficients), key=lambda x: abs(x[1]), reverse=True):
    direction = "↑ increases risk" if coef > 0 else "↓ decreases risk"
    print(f"  {name:<30} {coef:+.4f}   {direction}")

StatementMeta(, eb7d613a-2ffc-4468-b31c-e227f4514645, 9, Finished, Available, Finished, False)

--- Feature Coefficients (higher absolute value = more influential) ---
  avg_score                      -1.5352   ↓ decreases risk
  total_clicks                   -0.8217   ↓ decreases risk
  gender                         -0.2644   ↓ decreases risk
  studied_credits                +0.1894   ↑ increases risk
  num_of_prev_attempts           +0.0844   ↑ increases risk
  highest_education              +0.0771   ↑ increases risk
  age_band                       -0.0398   ↓ decreases risk


In [8]:
print("--- ML Sanity Checks ---")

preds = spark.table("gold_student_risk_predictions")

# Check probability range
bad_probs = preds.filter(
    (F.col("risk_probability") < 0) | (F.col("risk_probability") > 1)
).count()
print(f"{'✅' if bad_probs == 0 else '❌'} Risk probabilities in [0,1]: {bad_probs} violations")

# Check no nulls in key output columns
for col in ["student_id", "risk_probability", "predicted_at_risk"]:
    nulls = preds.filter(F.col(col).isNull()).count()
    print(f"{'✅' if nulls == 0 else '❌'} {col} nulls: {nulls}")

# AUC sanity
print(f"{'✅' if auc >= 0.65 else '⚠️  Low AUC — check features'} AUC-ROC: {auc:.4f}")

# Row count
count = preds.count()
print(f"{'✅' if count > 5000 else '❌'} Prediction row count: {count:,}")

StatementMeta(, eb7d613a-2ffc-4468-b31c-e227f4514645, 10, Finished, Available, Finished, False)

--- ML Sanity Checks ---
✅ Risk probabilities in [0,1]: 0 violations
✅ student_id nulls: 0
✅ risk_probability nulls: 0
✅ predicted_at_risk nulls: 0
✅ AUC-ROC: 0.8729
✅ Prediction row count: 5,726


In [9]:
# Score ALL students, not just the test set
all_students = model_df  # full dataset before split

all_predictions = model.transform(all_students) \
    .withColumn("prob_array", vector_to_array(F.col("probability"))) \
    .withColumn("risk_probability", F.round(F.col("prob_array")[1], 4)) \
    .withColumn("predicted_at_risk", F.col("prediction").cast("integer")) \
    .withColumn("_model_version", F.lit("logistic_regression_v1")) \
    .withColumn("_scored_at", F.current_timestamp()) \
    .select(
        "student_id", "avg_score", "total_clicks",
        "num_of_prev_attempts", "studied_credits",
        "is_at_risk", "predicted_at_risk", "risk_probability",
        "_model_version", "_scored_at"
    )

all_predictions.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_student_risk_predictions")

print(f"✅ Full predictions saved: {all_predictions.count():,} rows")

StatementMeta(, eb7d613a-2ffc-4468-b31c-e227f4514645, 11, Finished, Available, Finished, False)

✅ Full predictions saved: 28,785 rows
